# mlfoot-2026 - Notebook Storytelling

Objectif: raconter le cheminement complet, de la donnee brute a la prediction finale de la Coupe du Monde 2026.

Ton: passionne mais rigoureux.

## 1. Introduction - La Quete du Saint Graal (Predire 2026)

Nous voulons simuler 1 000 fois la competition pour estimer les probabilites de victoire.

Pitch: on combine l'histoire (Elo, performances internationales) et la qualite actuelle des joueurs (FBref) pour obtenir une vision plus robuste.

## 2. Etape 1 - La Collecte des Ingredients (Data Sourcing)

Sources principales:
- `data/raw/international_results.csv`: l'histoire des matchs internationaux.
- `data/raw/fifa_rankings.csv`: prestige et points FIFA.
- `data/raw/fbref/player_standard.csv` + `data/raw/squads.csv`: qualite intrinsique (joueurs -> selections).

Premier defi: agreger 4 saisons de stats clubs vers les selections nationales.

In [1]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path(".").resolve().parent
DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROC_DIR = DATA_DIR / "processed"

intl_path = RAW_DIR / "international_results.csv"
rank_path = RAW_DIR / "fifa_rankings.csv"
player_stats_path = RAW_DIR / "fbref" / "player_standard.csv"
squad_path = RAW_DIR / "squads.csv"

intl_df = pd.read_csv(intl_path)
rank_df = pd.read_csv(rank_path)
player_df = pd.read_csv(player_stats_path)
squad_df = pd.read_csv(squad_path)

print("international_results:", intl_df.shape)
print("fifa_rankings:", rank_df.shape)
print("player_stats:", player_df.shape)
print("squads:", squad_df.shape)

display(intl_df.head())
display(rank_df.head())
display(player_df.head())
display(squad_df.head())

international_results: (49287, 9)
fifa_rankings: (67472, 8)
player_stats: (11386, 25)
squads: (966, 7)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


,rank,country_full,country_abrv,total_points,previous_points,rank_change,confederation,rank_date
0,140.0,Brunei Darussalam,BRU,2.0,0.0,140,AFC,1992-12-31
1,33.0,Portugal,POR,38.0,0.0,33,UEFA,1992-12-31
2,32.0,Zambia,ZAM,38.0,0.0,32,CAF,1992-12-31
3,31.0,Greece,GRE,38.0,0.0,31,UEFA,1992-12-31
4,30.0,Algeria,ALG,39.0,0.0,30,CAF,1992-12-31


,league,season,team,Player,nation,pos,age,born,Playing Time_MP,Playing Time_Starts,...,Performance_G-PK,Performance_PK,Performance_PKatt,Performance_CrdY,Performance_CrdR,Per 90 Minutes_Gls,Per 90 Minutes_Ast,Per 90 Minutes_G+A,Per 90 Minutes_G-PK,Per 90 Minutes_G+A-PK
0,ENG-Premier League,2223,Arsenal,Aaron Ramsdale,ENG,GK,24,1998.0,38,38,...,0,0,0,1,0,0.00,0.00,0.00,0.00,0.00
1,ENG-Premier League,2223,Arsenal,Albert Sambi Lokonga,BEL,MF,22,1999.0,6,2,...,0,0,0,0,0,0.00,0.00,0.00,0.00,0.00
2,ENG-Premier League,2223,Arsenal,Ben White,ENG,DF,24,1997.0,38,36,...,2,0,0,5,0,0.06,0.15,0.21,0.06,0.21
3,ENG-Premier League,2223,Arsenal,Bukayo Saka,ENG,"FW,MF",20,2001.0,38,37,...,12,2,3,6,0,0.40,0.31,0.71,0.34,0.65
4,ENG-Premier League,2223,Arsenal,Cédric Soares,POR,"DF,MF",30,1991.0,2,0,...,0,0,0,0,0,0.00,0.00,0.00,0.00,0.00


,team,player,club,position,minutes,caps,goals
0,Portugal,Cristiano Ronaldo,Al Nassr,Attaquant,19350,215,135
1,Argentina,Lionel Messi,Inter Miami,Attaquant,16830,187,109
2,Belgium,Romelu Lukaku,Napoli,Attaquant,10350,115,85
3,England,Harry Kane,Bayern Munich,Attaquant,10080,112,78
4,Egypt,Mohamed Salah,Liverpool,Attaquant,9900,110,64


## 3. Etape 2 - L'Alchimie des Features (Engineering)

Piliers construits:
- Elo dynamique (forme historique)
- Forme recente avec decroissance exponentielle
- Importance des matchs

Pivot technique: transformation des stats joueurs en *Squad Features* (moyennes ponderees par le temps de jeu).

In [2]:
dataset_path = PROC_DIR / "match_dataset.csv"
df = pd.read_csv(dataset_path)

display(df.head())
print("match_dataset:", df.shape)
print("feature sample:", df.columns[:20].tolist())

missing = df.isna().mean().sort_values(ascending=False).head(10)
missing

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,is_friendly,...,away_squad_performance_g_pk,away_squad_performance_pk,away_squad_performance_pkatt,away_squad_performance_crdy,away_squad_performance_crdr,away_squad_per_90_minutes_gls,away_squad_per_90_minutes_ast,away_squad_per_90_minutes_g_a,away_squad_per_90_minutes_g_pk,away_squad_per_90_minutes_g_a_pk
0,2010-01-06,Bahrain,Hong Kong,4.0,0.0,AFC Asian Cup qualification,Riffa,Bahrain,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-01-06,China PR,Syria,0.0,0.0,AFC Asian Cup qualification,Hangzhou,China PR,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2010-01-06,Indonesia,Oman,1.0,2.0,AFC Asian Cup qualification,Jakarta,Indonesia,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2010-01-06,Kuwait,Australia,2.0,2.0,AFC Asian Cup qualification,Kuwait City,Kuwait,False,False,...,0.0,0.0,0.0,0.386152,0.079893,0.0,0.025526,0.025526,0.0,0.025526
4,2010-01-06,Lebanon,Vietnam,1.0,1.0,AFC Asian Cup qualification,Sidon,Lebanon,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


match_dataset: (10762, 81)
feature sample: ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral', 'is_friendly', 'match_id', 'result', 'match_importance', 'elo_home', 'elo_away', 'elo_diff', 'elo_home_post', 'elo_away_post', 'home_form_goals_scored_8', 'away_form_goals_scored_8']


away_squad_per_90_minutes_g_a_pk    0.796227
away_squad_performance_ast          0.796227
away_squad_minutes_total            0.796227
away_squad_caps                     0.796227
away_squad_season                   0.796227
away_squad_born                     0.796227
away_squad_playing_time_mp          0.796227
away_squad_playing_time_starts      0.796227
away_squad_playing_time_min         0.796227
away_squad_playing_time_90s         0.796227
dtype: float64

## 4. Etape 3 - Le Choix des Armes (Modelisation)

Candidats:
- XGBoost (le champion)
- Poisson (interpretabilite)
- Regression Logistique

Tournant de rigueur: Time Series Split + calibration des probabilites pour rester honnete sur la temporalite.

On montre les metriques finales (Log Loss, Accuracy).

In [3]:
metrics_path = PROC_DIR / "metrics.csv"
metrics_df = pd.read_csv(metrics_path)
metrics_df

,accuracy,log_loss,brier_score,model
0,0.573209,0.894157,0.529044,challenger_poisson
1,0.576843,0.901544,0.533583,challenger_logistic
2,0.576843,0.906733,0.535699,champion_xgboost
3,0.574247,0.983740,0.564751,challenger_lightgbm


## 5. Etape 4 - Nettoyer le Bruit (Feature Importance)

Decouverte: `city` et `country` ajoutent du bruit et prennent beaucoup de place dans les features importance (geographie non predictive).

On conserve un modele plus strategique: Elo, FIFA points, temps de jeu FBref.

Visualisation: Top 20 des features finales.

In [4]:
import joblib
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import colorsys
from pathlib import Path

# --- Chargement modèle ---
model_path = PROC_DIR / "models" / "champion_xgboost.pkl"
model = joblib.load(model_path)

# --- Extraction pipeline (CalibratedClassifierCV ou direct) ---
if hasattr(model, "calibrated_classifiers_"):
    first_fold = model.calibrated_classifiers_[0].estimator
else:
    first_fold = model

preprocessor = first_fold.steps[0][1]
xgb_model    = first_fold.steps[-1][1]

try:
    feature_names = preprocessor.get_feature_names_out()
except Exception:
    feature_names = [f"feature_{i}" for i in range(len(xgb_model.feature_importances_))]

# --- Nettoyage des noms de features (retire les préfixes sklearn type "num__", "cat__") ---
def clean_name(name: str) -> str:
    if "__" in name:
        name = name.split("__", 1)[1]
    return name.replace("_", " ").title()

# --- DataFrame ---
importances = xgb_model.feature_importances_
min_len = min(len(feature_names), len(importances))

df_imp = (
    pd.DataFrame({
        "Feature":    [clean_name(f) for f in feature_names[:min_len]],
        "Feature_raw": feature_names[:min_len],
        "Importance": importances[:min_len],
    })
    .sort_values("Importance", ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# Tri croissant pour le bar horizontal (top en haut avec autorange=reversed)
df_plot = df_imp.sort_values("Importance", ascending=True).reset_index(drop=True)

# --- Palette gradient (même logique que WC 2026) ---
n = len(df_plot)

def oklch_approx_to_hex(l, c, h_deg):
    h = h_deg / 360
    s = min(c * 2.5, 1.0)
    r, g, b = colorsys.hls_to_rgb(h, l, s)
    return f"#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}"

# Gradient du bas (faible importance, bleu froid) vers le haut (forte, teal chaud)
hex_colors = [
    oklch_approx_to_hex(0.42 + 0.30 * (i / max(n - 1, 1)), 0.14, 200 - 25 * (i / max(n - 1, 1)))
    for i in range(n)
]

# --- Seuil top 5 ---
threshold = df_plot["Importance"].nlargest(5).min()

# --- Figure ---
fig = go.Figure()

fig.add_trace(go.Bar(
    x=df_plot["Importance"],
    y=df_plot["Feature"],
    orientation="h",
    marker=dict(
        color=hex_colors,
        line=dict(width=0),
        cornerradius=5,
    ),
    text=[f"{v:.4f}" for v in df_plot["Importance"]],
    textposition="outside",
    textfont=dict(size=11, family="Inter, sans-serif", color="#7a7974"),
    hovertemplate="<b>%{y}</b><br>Importance : <b>%{x:.5f}</b><extra></extra>",
    cliponaxis=False,
))

# --- Ligne de seuil top 5 ---
fig.add_vline(
    x=threshold,
    line_dash="dot",
    line_color="#7a7974",
    line_width=1.5,
    annotation_text="Seuil Top 5",
    annotation_position="top left",
    annotation_font=dict(size=11, color="#7a7974"),
)

# --- Étoile sur la feature #1 ---
top_feature = df_plot.iloc[-1]
fig.add_annotation(
    x=top_feature["Importance"],
    y=top_feature["Feature"],
    text="⭐",
    showarrow=False,
    xanchor="left",
    xshift=52,
    font=dict(size=15),
)

# --- Layout ---
max_imp = df_plot["Importance"].max()

fig.update_layout(
    title=dict(
        text=(
            "<b>Top 20 — Feature Importances · XGBoost</b><br>"
            "<span style='font-size:13px;color:#7a7974'>"
            "Gain moyen normalisé · Modèle champion calibré</span>"
        ),
        font=dict(size=20, family="Inter, sans-serif", color="#28251d"),
        x=0,
        xanchor="left",
        pad=dict(b=20),
    ),
    xaxis=dict(
        title=dict(
            text="Importance (gain normalisé)",
            font=dict(size=13, color="#7a7974"),
        ),
        gridcolor="#dcd9d5",
        gridwidth=1,
        zeroline=False,
        range=[0, max_imp * 1.20],
        tickfont=dict(size=11, family="Inter, sans-serif", color="#7a7974"),
    ),
    yaxis=dict(
        title=None,
        tickfont=dict(size=12, family="Inter, sans-serif", color="#28251d"),
        automargin=True,    # évite la troncature des longs noms
    ),
    plot_bgcolor="#f7f6f2",
    paper_bgcolor="#f9f8f5",
    margin=dict(l=10, r=70, t=95, b=60),
    height=580,
    width=820,
    showlegend=False,
    bargap=0.28,
    font=dict(family="Inter, sans-serif"),
    hoverlabel=dict(
        bgcolor="#1c1b19",
        font=dict(color="#cdccca", size=13, family="Inter, sans-serif"),
        bordercolor="#393836",
    ),
)

# --- Watermark ---
fig.add_annotation(
    text="mlfoot · XGBoost Champion Model",
    xref="paper", yref="paper",
    x=1, y=-0.10,
    xanchor="right",
    font=dict(size=10, color="#bab9b4"),
    showarrow=False,
)

fig.show()
# fig.write_image("feature_importance_xgb.png", scale=2)

## 6. Etape 5 - La Grande Simulation (Monte Carlo)

Processus: 1 000 tournois virtuels, des poules a la finale.

Resultats attendus: Top 5 (Argentine, France, Belgique, Bresil, Espagne).

Analyse: Belgique en dark horse, Argentine sur le trone.

In [5]:
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path
import colorsys

# --- Chargement des données ---
sim_path = BASE_DIR / "data" / "processed" / "wc2026_simulation.csv"
sim_df = pd.read_csv(sim_path)

top10 = sim_df.sort_values("win_prob", ascending=False).head(10).reset_index(drop=True)

# --- Palette de couleurs gradient personnalisée (du plus fort au plus faible) ---
n = len(top10)
colors = [
    f"oklch({0.45 + 0.35 * (1 - i / (n - 1)):.2f} 0.15 {192 - 30 * i / (n - 1):.0f})"
    for i in range(n)
 ]
# Fallback hex si oklch non supporte dans Plotly

def oklch_to_hex(l, c, h_deg):
    """Approximation oklch -> RGB hex via HLS pour Plotly."""
    h = h_deg / 360
    s = min(c * 2.5, 1.0)
    r, g, b = colorsys.hls_to_rgb(h, l, s)
    return f"#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}"

hex_colors = [
    oklch_to_hex(0.45 + 0.35 * (1 - i / (n - 1)), 0.15, 192 - 30 * i / (n - 1))
    for i in range(n)
 ]

# --- Construction du graphique ---
fig = go.Figure()

fig.add_trace(go.Bar(
    x=top10["win_prob"],
    y=top10["team"],
    orientation="h",
    marker=dict(
        color=hex_colors,
        line=dict(width=0),
        cornerradius=6,
    ),
    text=[f"{v:.1%}" for v in top10["win_prob"]],
    textposition="outside",
    textfont=dict(size=13, family="Inter, sans-serif", color="#28251d"),
    hovertemplate="<b>%{y}</b><br>Probabilite de titre : <b>%{x:.2%}</b><extra></extra>",
    cliponaxis=False,
))

# --- Annotation du leader ---
top_team = top10.iloc[0]["team"]
top_prob = top10.iloc[0]["win_prob"]
fig.add_annotation(
    x=top_prob,
    y=top_team,
    text="🏆",
    showarrow=False,
    xanchor="left",
    xshift=55,
    font=dict(size=18),
    yshift=0,
 )

# --- Layout ---
title_text = (
    "<b>Top 10 — Probabilite de Victoire Finale</b><br>"
    "<span style='font-size:13px;color:#7a7974'>Coupe du Monde 2026 · Simulation Monte Carlo</span>"
 )
fig.update_layout(
    title=dict(
        text=title_text,
        font=dict(size=20, family="Inter, sans-serif", color="#28251d"),
        x=0,
        xanchor="left",
        pad=dict(b=20),
    ),
    xaxis=dict(
        title=dict(
            text="Probabilite de titre",
            font=dict(size=13, color="#7a7974"),
        ),
        tickformat=".0%",
        gridcolor="#dcd9d5",
        gridwidth=1,
        zeroline=False,
        range=[0, top10["win_prob"].max() * 1.22],
        tickfont=dict(size=12, family="Inter, sans-serif", color="#7a7974"),
    ),
    yaxis=dict(
        title=None,
        autorange="reversed",
        tickfont=dict(size=13, family="Inter, sans-serif", color="#28251d"),
    ),
    plot_bgcolor="#f7f6f2",
    paper_bgcolor="#f9f8f5",
    margin=dict(l=20, r=60, t=90, b=60),
    height=460,
    width=780,
    showlegend=False,
    bargap=0.35,
    font=dict(family="Inter, sans-serif"),
    hoverlabel=dict(
        bgcolor="#1c1b19",
        font=dict(color="#cdccca", size=13, family="Inter, sans-serif"),
        bordercolor="#393836",
    ),
 )

# --- Watermark ---
fig.add_annotation(
    text="WC 2026 · Monte Carlo Simulation",
    xref="paper", yref="paper",
    x=1, y=-0.12,
    xanchor="right",
    font=dict(size=10, color="#bab9b4"),
    showarrow=False,
 )

fig.show()

In [6]:
import plotly.express as px

fig = px.histogram(
    sim_df,
    x="win_prob",
    nbins=30,
    title="Distribution des probabilites de victoire",
    labels={"win_prob": "Probabilite"},
)
fig.show()

In [7]:
sim_df.sort_values("win_prob", ascending=False).head(20)

,team,qual_prob,r16_prob,qf_prob,sf_prob,final_prob,win_prob
0,Argentina,0.957,0.749,0.534,0.347,0.226,0.128
1,France,0.937,0.682,0.472,0.316,0.190,0.113
2,Brazil,0.933,0.700,0.471,0.297,0.184,0.112
3,Belgium,0.974,0.763,0.531,0.342,0.204,0.105
4,England,0.878,0.632,0.388,0.221,0.134,0.080
5,Spain,0.919,0.690,0.432,0.267,0.149,0.078
6,Portugal,0.891,0.651,0.389,0.231,0.127,0.074
7,Germany,0.919,0.662,0.414,0.246,0.130,0.061
8,Uruguay,0.804,0.519,0.289,0.134,0.064,0.032
9,Netherlands,0.901,0.614,0.342,0.186,0.082,0.029


In [8]:
heat = sim_df.set_index("team")[["qual_prob", "r16_prob", "qf_prob", "sf_prob", "final_prob", "win_prob"]]
heat.columns = ["Poules", "R16", "QF", "SF", "Finale", "Vainqueur"]

fig = px.imshow(
    heat.sort_values("Vainqueur", ascending=False).head(15),
    title="Chances de survie par etape",
    aspect="auto",
    color_continuous_scale="RdYlGn",
)
fig.show()

![Confusion matrix](../data/processed/plots/cm_champion_xgboost.png)   